## Year-wise Leave Policy RAG
1. Load every PDF in `Leave_Policy` and chunk it, tagging each chunk with its year
2. Embed the chunks and store them in Pinecone
3. Answer single-year and multi-year comparison questions (with memory)

In [3]:
import os
import re
import json
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

load_dotenv()

FOLDER = "Leave_Policy"

with open(os.path.join(FOLDER, "manifest.json"), encoding="utf-8") as f:
    manifest = {m["year"]: m for m in json.load(f)}

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""],
)

chunk_records = []  # one dict per chunk: year, file, text
for year, meta in sorted(manifest.items()):
    pages = PyPDFLoader(os.path.join(FOLDER, meta["file"])).load()
    text = "\n".join(p.page_content for p in pages)

    chunks = splitter.split_text(text)

    for chunk in chunks:
        # Prefix the year so every chunk is self-describing
        chunk_records.append({
            "year": year,
            "file": meta["file"],
            "text": f"[Leave Policy {year}] {chunk}",
        })

print("Years loaded:", sorted(manifest))
print("Total chunks:", len(chunk_records))

Years loaded: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
Total chunks: 138


In [4]:
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

pc = Pinecone(api_key=os.getenv("PineCone_API_Key"))


def get_index(name):
    if name not in pc.list_indexes().names():
        pc.create_index(
            name=name,
            dimension=384,
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1"),
        )
    return pc.Index(name)


index = get_index(os.getenv("Pinecone_Index_Name"))
memory_index = get_index(os.getenv("Pinecone_Memory_Index_Name", "employeeleavepolicy-longmemory"))

# Own namespace so these records don't mix with the single-document data
POLICY_NS = "leave-policy-yearly"

vectors = model.encode([r["text"] for r in chunk_records])

records = [
    {
        "id": f"lp-{r['year']}-{i}",
        "values": vectors[i].tolist(),
        "metadata": {"text": r["text"], "year": r["year"], "source": r["file"], "chunk_index": i},
    }
    for i, r in enumerate(chunk_records)
]

# Re-runnable: clear old records of this namespace first
try:
    index.delete(delete_all=True, namespace=POLICY_NS)
except Exception:
    pass  # namespace doesn't exist yet

for start in range(0, len(records), 100):
    index.upsert(vectors=records[start:start + 100], namespace=POLICY_NS)
    print(f"Inserted {min(start + 100, len(records))}/{len(records)}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2920.05it/s]


Inserted 100/138
Inserted 138/138


In [ ]:
import uuid
import anthropic

client = anthropic.Anthropic(api_key=os.getenv("Claude_API_KEY"))

def ask_claude(prompt):
    """Claude's text answer, or None if Claude declined."""
    response = client.beta.messages.create(
        model="claude-opus-5-5",
        max_tokens=16000,
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",  # if Claude declines, the API retries on a fallback model
        messages=[{"role": "user", "content": prompt}],
    )
    if response.stop_reason == "refusal":
        return None
    return next(b.text for b in response.content if b.type == "text")

MEMORY_THRESHOLD = 0.90
TOP_K_PER_YEAR = 8
MIN_YEAR, MAX_YEAR = min(manifest), max(manifest)


def extract_years(q):
    """Years in the question; 'X to Y' / 'between X and Y' expands to every year in the range."""
    q_low = q.lower()
    years = {int(y) for y in re.findall(r"\b(20\d{2})\b", q)}
    for a, b in re.findall(r"\b(20\d{2})\s*(?:to|-|through|and till)\s*(20\d{2})\b", q_low):
        years.update(range(int(a), int(b) + 1))
    return sorted(years)


def is_comparison(q, years):
    words = ("compare", "comparison", "difference", "differ", "vs", "versus", "change", "changed")
    return len(years) > 1 or any(w in q.lower() for w in words)


def retrieve(q_vector, years):
    """Top chunks per requested year (metadata filter), or across all years if none given."""
    ctx = {}
    if not years:
        res = index.query(vector=q_vector, top_k=TOP_K_PER_YEAR, include_metadata=True, namespace=POLICY_NS)
        for m in res.matches:
            ctx.setdefault(int(m.metadata["year"]), []).append(m.metadata["text"])
        return ctx
    for y in years:
        res = index.query(
            vector=q_vector,
            top_k=TOP_K_PER_YEAR,
            include_metadata=True,
            namespace=POLICY_NS,
            filter={"year": {"$eq": y}},
        )
        ctx[y] = [m.metadata["text"] for m in res.matches]
    return ctx


def build_prompt(q, ctx, compare):
    sections = "\n\n".join(
        f"=== Leave Policy {y} ===\n" + "\n".join(chunks) for y, chunks in sorted(ctx.items())
    )
    if compare:
        task = (
            "Compare the leave policies of the years provided. Present a clear comparison table "
            "(one column per year) of each leave type, then summarise what increased, decreased, "
            "or stayed the same."
        )
    else:
        task = "Answer the question using the leave policy of the year provided."
    return f"""
You are a professional employee leave policy assistant.
{task}

Rules:
- Use ONLY the policy text below; do not invent facts or change numbers.
- Always mention the year for every figure.
- If a year or detail is missing from the text, say so clearly.

Question:
{q}

Policy text:
{sections}

Answer:
"""


def ask(q):
    q_vector = model.encode(q).tolist()
    years = extract_years(q)
    years_key = ",".join(map(str, years)) or "all"

    # 1. Memory first; years_key stops '2025 vs 2026' reusing a '2024 vs 2026' answer
    mem = memory_index.query(
        vector=q_vector, top_k=1, include_metadata=True,
        filter={"years_key": {"$eq": years_key}},
    )
    if mem.matches and mem.matches[0].score >= MEMORY_THRESHOLD:
        hit = mem.matches[0]
        print(f"[Memory HIT] score={hit.score:.3f}, matched: {hit.metadata['question']}")
        return hit.metadata["answer"]

    # 2. Normal RAG
    print(f"[Memory MISS] years={years or 'not specified'}")
    missing = [y for y in years if y not in manifest]
    if missing:
        return f"No leave policy available for {missing}. Available years: {MIN_YEAR}-{MAX_YEAR}."

    ctx = retrieve(q_vector, years)
    prompt = build_prompt(q, ctx, is_comparison(q, years))
    answer = ask_claude(prompt)
    if answer is None:
        return "Sorry, I can't answer that question."

    # 3. Save to memory
    memory_index.upsert(vectors=[{
        "id": f"mem-{uuid.uuid4()}",
        "values": q_vector,
        "metadata": {"question": q, "answer": answer, "years_key": years_key},
    }])
    return answer

In [ ]:
print(ask("How many casual leaves are there in 2026?"))

In [ ]:
print(ask("Compare leave policy of 2015 and 2026"))

In [ ]:
print(ask("What is the difference in sick leave between 2020, 2023 and 2026?"))